# Codex

Codex backend

In [ ]:
#| default_exp codex

## Imports

In [ ]:
#| export
from fastcore.utils import *

In [ ]:
#| hide
from fastcore.test import test_eq
from cachy import enable_cachy

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
_CLIENT_ID  = 'app_EMoamEEZ73f0CkXaXp7hrann'
_TOKEN_URL  = 'https://auth.openai.com/oauth/token'
_SKEW_SECS  = 60

In [ ]:
_DEFAULT_AUTH_FILE = '~/.codex/auth.json'
def _auth_file(): return Path(os.environ.get('CODEX_AUTH_FILE', _DEFAULT_AUTH_FILE)).expanduser()

In [ ]:
def _jwt_exp(token):
    parts = token.split('.')
    if len(parts)<2: return None
    pad = parts[1] + '='*(-len(parts[1])%4)
    claims = json.loads(base64.urlsafe_b64decode(pad).decode())
    exp = claims.get('exp')
    return int(exp) if isinstance(exp,(int,float)) else None

In [ ]:
def _expired(token):
    exp = _jwt_exp(token)
    return exp is None or time.time() >= exp - _SKEW_SECS

In [ ]:
def _read_auth():
    f = _auth_file()
    return json.loads(f.read_text()) if f.exists() else {}


In [ ]:
def _write_auth(d): _auth_file().write_text(json.dumps(d, indent=2))


In [ ]:
def _refresh(refresh_token):
    body = dict(client_id=_CLIENT_ID, grant_type='refresh_token', refresh_token=refresh_token, scope='openid profile email')
    r = httpx.post(_TOKEN_URL, json=body)
    r.raise_for_status()
    return r.json()

In [ ]:
def codex_auth(force_refresh=False):
    "Return a valid codex access token, refreshing via OAuth if expired or `force_refresh`."
    data = _read_auth()
    tokens = data.get('tokens', {})
    at, rt = tokens.get('access_token'), tokens.get('refresh_token')
    if at and not force_refresh and not _expired(at): return at
    if not rt: raise ValueError(f"No refresh_token in {_auth_file()}; run `codex login` first")
    new = _refresh(rt)
    tokens['access_token']  = new['access_token']
    tokens['id_token']      = new.get('id_token', tokens.get('id_token'))
    tokens['refresh_token'] = new.get('refresh_token', rt)
    data['tokens'] = tokens
    _write_auth(data)
    return tokens['access_token']

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()